# 🍽️ Restaurant Growth Potential & Strategic Classification
### End-to-End Machine Learning, Unit Economics & Statistical Reliability Engine
**Auckland Hospitality Assets Portfolio Analysis (SkyCity Initiative)**

---

## 📌 Project Overview
The hospitality industry faces a major structural challenge: **the delivery growth paradox**. While restaurants can rapidly scale gross order volume on third-party aggregators (Uber Eats, DoorDash), steep platform commissions (25%–35%) often turn incremental orders into net operating losses. 

This notebook provides a complete analytical and machine learning workflow to:
1. **Audit Data Quality** across 1,696 Auckland restaurant records.
2. **Engineer Multi-Channel Unit Economics** (channel margins, aggregator dependence, delivery cost efficiency, Herfindahl channel balance).
3. **Apply Dimensionality Reduction (PCA)** with Yeo-Johnson power transformations.
4. **Discover Operational Archetypes** via K-Means clustering and Hungarian bipartite matching.
5. **Compute a Multi-Pillar Growth Potential Index (GPI)** to rank venues across Growth, Cost, Channel, and Logistics dimensions.
6. **Statistically Validate Separation** using non-parametric Kruskal-Wallis ANOVA and 1,000-sample bootstrap confidence intervals.
7. **Simulate Strategic What-If Scenarios** to quantify financial upside from channel rebalancing.

In [ ]:
# Core Scientific & Data Processing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn Machine Learning Stack
from sklearn.preprocessing import PowerTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.optimize import linear_sum_assignment

# Local Analytical Engines
from pipeline import (
    load_data, engineer_features, preprocess, fit_pca,
    fit_cluster, label_clusters, cluster_quality, k_scan,
    CLUSTER_FEATURES, THEMES, ARCHETYPES, ARCHETYPE_DESC
)
from scoring import (
    score_and_classify, add_scores, compute_gpi, classify,
    DEFAULT_WEIGHTS, PILLARS
)
from reliability import (
    data_quality_report, quality_score, cluster_separation_tests,
    bootstrap_gpi_ci, classification_confidence
)

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10
print("✅ All modules and dependencies imported successfully.")

---
## 1. Data Ingestion & Exploratory Data Analysis (EDA)

We load the verified Auckland dining dataset (`data/skycity_restaurants.csv`), clean column headers, validate schema constraints, and explore key distributions.

In [ ]:
# Load and validate raw dataset
data_path = "data/skycity_restaurants.csv"
df_raw = load_data(data_path)

print(f"Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Unique Restaurants: {df_raw['RestaurantID'].nunique()}")
print(f"Cuisine Types: {df_raw['CuisineType'].nunique()} | Subregions: {df_raw['Subregion'].nunique()}")

# Display sample records
df_raw[['RestaurantID', 'RestaurantName', 'CuisineType', 'Subregion', 'Segment', 'MonthlyOrders', 'AOV', 'GrowthFactor']].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df_raw['MonthlyOrders'], kde=True, ax=axes[0], color='#0E7C86')
axes[0].set_title('Distribution of Monthly Orders')
axes[0].set_xlabel('Monthly Orders')

sns.histplot(df_raw['AOV'], kde=True, ax=axes[1], color='#B8860B')
axes[1].set_title('Distribution of Average Order Value (AOV $)')
axes[1].set_xlabel('AOV ($)')

sns.histplot(df_raw['GrowthFactor'], kde=True, ax=axes[2], color='#1B9E77')
axes[2].axvline(1.0, color='red', linestyle='--', label='Neutral Growth (1.0x)')
axes[2].set_title('Distribution of Growth Factor')
axes[2].set_xlabel('Monthly Growth Multiplier')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 2. Automated Data Quality & Reliability Audit

Before running analytical pipelines, an automated audit scans for missing data, Tukey's IQR statistical outliers, Fisher-Pearson distribution skewness, and entity integrity.

In [ ]:
# Generate comprehensive data quality report
dq_report = data_quality_report(df_raw)
dq_score = quality_score(dq_report)

print("=" * 60)
print(f"📋 AUTOMATED DATA QUALITY AUDIT SCORE: {dq_score} / 100")
print("=" * 60)
print(f"• Total Rows Audited: {dq_report['rows']:,}")
print(f"• Mean Feature Completeness: {dq_report['mean_completeness']:.1%}")
print(f"• Total IQR Outliers Flagged: {dq_report['total_outliers']}")
print(f"• Highly Skewed Features (|skew| > 2.0): {len(dq_report['highly_skewed'])}")
print(f"• Duplicate Restaurant IDs: {dq_report['duplicate_ids']}")
print(f"• Negative Revenue Violations: {sum(dq_report['negative_revenue'].values())}")
print("=" * 60)

---
## 3. Feature Synthesis & Multi-Channel Unit Economics

Hospitality venues operate four distinct order fulfillment channels:
1. **In-Store Dining**
2. **Uber Eats** (Third-party aggregator)
3. **DoorDash** (Third-party aggregator)
4. **Self-Delivery** (Proprietary logistics)

We engineer metrics that expose true net profit margins by channel, aggregator dependence, and channel concentration (Herfindahl-Hirschman index derivative).

In [ ]:
# Compute synthesized financial and logistical features
df_feat = engineer_features(df_raw)

# Portfolio-wide totals
tot_rev = df_feat['TotalRevenue'].sum()
tot_profit = df_feat['TotalNetProfit'].sum()
port_margin = tot_profit / tot_rev

print(f"Total Monthly Portfolio Revenue: ${tot_rev:,.2f}")
print(f"Total Monthly Portfolio Net Profit: ${tot_profit:,.2f}")
print(f"Overall Portfolio Margin: {port_margin:.2%}")
print(f"Average Aggregator Dependence: {df_feat['AggregatorDependence'].mean():.1%}")
print(f"Average Aggregator Commission Rate: {df_feat['CommissionRate'].mean():.1%}")

In [ ]:
# Analyze the Aggregator Profitability Paradox
agg_loss_mask = df_feat['AggregatorMargin'] < 0
pct_agg_loss = agg_loss_mask.mean() * 100
total_agg_loss = df_feat.loc[agg_loss_mask, ['UberEatsNetProfit', 'DoorDashNetProfit']].sum().sum()

sd_wins = (df_feat['SD_Margin'] > df_feat['AggregatorMargin']).mean() * 100
margin_gap = (df_feat['SD_Margin'] - df_feat['AggregatorMargin']).mean() * 100

print(f"⚠️ Aggregator Loss Incidence: {pct_agg_loss:.1f}% of restaurants LOSE money on third-party aggregators.")
print(f"💸 Estimated Monthly Losses on Aggregators: ${abs(total_agg_loss):,.2f}/month across the cohort.")
print(f"🚀 Self-Delivery Advantage: Self-delivery outperforms aggregators for {sd_wins:.1f}% of restaurants.")
print(f"📈 Average Margin Premium for Self-Delivery: +{margin_gap:.1f} percentage points.")

# Visualizing Margin Differences
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(df_feat['InStoreMargin'], label='In-Store Margin', fill=True, alpha=0.3, color='#1B9E77')
sns.kdeplot(df_feat['SD_Margin'], label='Self-Delivery Margin', fill=True, alpha=0.3, color='#0E7C86')
sns.kdeplot(df_feat['AggregatorMargin'], label='Aggregator Margin (UE + DD)', fill=True, alpha=0.3, color='#C0392B')
ax.axvline(0, color='black', linestyle=':', label='Breakeven (0% Margin)')
ax.set_title('Channel Profitability Comparison (In-Store vs. Self-Delivery vs. Aggregators)')
ax.set_xlabel('Net Operating Margin')
ax.set_ylabel('Density')
ax.legend()
plt.show()

---
## 4. Dimensionality Reduction & PCA Eigenspace

We apply **Yeo-Johnson power transformations** to stabilize non-normal feature distributions, scale categorical dummies with shrinkage weighting ($\omega_{	ext{cat}} = 0.35$), and fit **Principal Component Analysis (PCA)**.

In [ ]:
# Preprocess features & apply Yeo-Johnson transformation
X_processed, feature_names = preprocess(df_feat, include_cat=True, cat_weight=0.35)

# Fit PCA targeting >= 85% cumulative variance
pca, Z_pca, n_components = fit_pca(X_processed, var_target=0.85)

cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
print(f"Retained {n_components} components explaining {cum_var[n_components-1]:.1f}% of total variance.")

# Scree Plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 11), cum_var[:10], marker='o', color='#0E7C86', linewidth=2)
ax.axhline(85, color='red', linestyle='--', label='85% Variance Threshold')
ax.set_title('PCA Scree Plot (Cumulative Explained Variance)')
ax.set_xlabel('Principal Component Index')
ax.set_ylabel('Cumulative Variance (%)')
ax.set_xticks(range(1, 11))
ax.legend()
plt.show()

---
## 5. Unsupervised Clustering & Strategic Archetyping

We evaluate cluster configurations using Silhouette Score and Davies-Bouldin Index, fit $k$-Means with $k=5$, and use the **Kuhn-Munkres (Hungarian) algorithm** to match clusters into five validated operational archetypes:
1. **High-Growth / High-Risk**
2. **Stable Local Performers**
3. **Scalable Self-Delivery Leaders**
4. **Aggregator-Dependent Low Margin**
5. **Overextended, Low Return**

In [ ]:
# Scan k from 2 to 8
scan_results = k_scan(Z_pca[:, :n_components], ks=range(2, 9))
print("Clustering Evaluation Across k:")
print(scan_results[['k', 'Silhouette', 'Davies-Bouldin', 'Inertia']].to_string(index=False))

# Fit optimal k=5 model
k_optimal = 5
cluster_labels = fit_cluster(Z_pca[:, :n_components], algo="K-Means", k=k_optimal, seed=42)
archetype_map = label_clusters(df_feat, cluster_labels)

# Assign to dataframe
df_feat['ClusterID'] = cluster_labels
df_feat['Cluster'] = [archetype_map[c] for c in cluster_labels]

print("
Cluster Distribution:")
print(df_feat['Cluster'].value_counts())

In [ ]:
# Project to 2D PCA space and visualize
plt.figure(figsize=(11, 7))
palette = {
    "High-Growth / High-Risk": "#E6A100",
    "Stable Local Performers": "#1B9E77",
    "Scalable Self-Delivery Leaders": "#0E7C86",
    "Aggregator-Dependent Low Margin": "#C0392B",
    "Overextended, Low Return": "#7570B3"
}

for archetype, color in palette.items():
    subset = df_feat[df_feat['Cluster'] == archetype]
    indices = subset.index
    plt.scatter(Z_pca[indices, 0], Z_pca[indices, 1], label=f"{archetype} (n={len(subset)})", 
                alpha=0.6, s=35, color=color)

plt.title('Restaurant Behavioral Archetypes (PCA 2D Projection)', fontsize=13, fontweight='bold')
plt.xlabel('Principal Component 1 (Size & Channel Scale)')
plt.ylabel('Principal Component 2 (Margin vs. Cost Pressure)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 6. Growth Potential Index (GPI) & Strategic Tri-Classification

The **Growth Potential Index (GPI)** is a 0–100 percentile-ranked metric spanning 4 core pillars:
- **Growth Signals (30%):** Growth factor, scale volume, and AOV.
- **Cost Resilience (30%):** Inverted cost rate and net margin.
- **Channel Balance (20%):** Channel balance index, inverted aggregator dependence, aggregator margin.
- **Logistics Scalability (20%):** Self-delivery margin, delivery cost efficiency, and expansion headroom.

Restaurants are classified into:
- **Optimize:** Ready for expansion capital.
- **Rebalance channels:** Structurally sound, but bleeding margin on third-party aggregators.
- **Hold / Stabilize:** Bottom 35% GPI; operational turnaround required before investing.

In [ ]:
# Execute Scoring and Classification
df_scored, hold_threshold = score_and_classify(df_feat, DEFAULT_WEIGHTS, hold_q=0.35, agg_margin_floor=0.0)

print(f"GPI Score Distribution: Mean = {df_scored['GPI'].mean():.1f} | Median = {df_scored['GPI'].median():.1f} | Std = {df_scored['GPI'].std():.1f}")
print(f"Hold / Stabilize Cutoff Threshold: {hold_threshold:.1f} GPI points
")

print("Strategic Classification Breakdown:")
strat_counts = df_scored['Strategy'].value_counts()
for strat, cnt in strat_counts.items():
    print(f"• {strat}: {cnt} venues ({cnt/len(df_scored)*100:.1f}%)")

In [ ]:
# Archetype Profile Comparison Table
archetype_summary = df_scored.groupby('Cluster').agg(
    Venues=('RestaurantID', 'count'),
    Mean_GPI=('GPI', 'mean'),
    Net_Margin=('NetMargin', lambda x: f"{x.mean():.1%}"),
    Agg_Share=('AggregatorDependence', lambda x: f"{x.mean():.1%}"),
    SD_Margin=('SD_Margin', lambda x: f"{x.mean():.1%}"),
    Growth_Factor=('GrowthFactor', lambda x: f"{x.mean():.3f}x")
).loc[list(palette.keys())]

display(archetype_summary)

# Boxplot of GPI across Archetypes
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_scored, x='Cluster', y='GPI', palette=palette, order=list(palette.keys()))
plt.title('Growth Potential Index (GPI) Distribution by Operational Archetype')
plt.xticks(rotation=15, ha='right')
plt.ylabel('GPI Score (0-100)')
plt.tight_layout()
plt.show()

---
## 7. Statistical Validation & Hypothesis Testing

To prove that the archetypes represent statistically distinct operational regimes rather than random partitions:
1. **Kruskal-Wallis Non-Parametric ANOVA:** Tests for global divergence across groups.
2. **Mann-Whitney $U$ Tests:** Measures pairwise differences and effect sizes.
3. **Bootstrap 95% Confidence Intervals:** Quantifies ranking uncertainty over 200 resampling iterations.

In [ ]:
# Run Kruskal-Wallis & Mann-Whitney tests on GPI separation
sep_table, kw_stats = cluster_separation_tests(df_scored, metric="GPI")

print("=" * 60)
print("📊 NON-PARAMETRIC HYPOTHESIS TESTING (KRUSKAL-WALLIS)")
print("=" * 60)
print(f"• H-Statistic: {kw_stats['H-statistic']:.2f}")
print(f"• p-value: {kw_stats['p-value']:.2e} (p < 1e-15)")
print(f"• Null Hypothesis Rejected: {kw_stats['Significant']} (Archetypes are statistically distinct)")
print("=" * 60)

# Display sample of pairwise Mann-Whitney tests
display(sep_table.head(6))

In [ ]:
# Compute Bootstrap Confidence Intervals
ci_results = bootstrap_gpi_ci(df_scored, DEFAULT_WEIGHTS, n_boot=200, seed=42)

df_scored['GPI_lo'] = ci_results['GPI_lo']
df_scored['GPI_hi'] = ci_results['GPI_hi']
df_scored['Confidence'] = ci_results['Confidence']

print(f"Mean 95% Confidence Interval Width: {ci_results['CI_width'].mean():.2f} GPI points")
print(f"Average Reliability Confidence Score: {ci_results['Confidence'].mean():.1f}%")

# Visualize sample confidence intervals
sample_venues = df_scored.sample(15, random_state=42).sort_values('GPI')
plt.figure(figsize=(10, 6))
y_pos = range(len(sample_venues))
plt.errorbar(sample_venues['GPI'], y_pos, 
             xerr=[sample_venues['GPI'] - sample_venues['GPI_lo'], sample_venues['GPI_hi'] - sample_venues['GPI_GPI']], 
             fmt='o', color='#0E7C86', ecolor='#B8860B', elinewidth=2, capsize=4)
plt.yticks(y_pos, sample_venues['RestaurantName'])
plt.xlabel('Growth Potential Index (GPI) with 95% Bootstrap Confidence Intervals')
plt.title('Rank Uncertainty Quantification (Sample of 15 Auckland Restaurants)')
plt.tight_layout()
plt.show()

---
## 8. Strategic What-If Simulation

What happens if restaurants in the **Rebalance channels** category shift 20% of their aggregator volume to self-delivery or direct channels at an 80% retention rate?

We model the monthly financial recovery.

In [ ]:
# Model order migration for Rebalance candidates
rebalance_df = df_scored[df_scored['Strategy'] == 'Rebalance channels'].copy()

migration_pct = 0.20   # 20% shifted
retention_rate = 0.80  # 80% retained

# Order shift
rebalance_df['ShiftedOrders'] = (rebalance_df['UberEatsOrders'] + rebalance_df['DoorDashOrders']) * migration_pct * retention_rate
rebalance_df['MarginAdvantage'] = (rebalance_df['SD_Margin'] - rebalance_df['AggregatorMargin']).clip(lower=0)
rebalance_df['MonthlyRecovery'] = rebalance_df['ShiftedOrders'] * rebalance_df['AOV'] * rebalance_df['MarginAdvantage']

total_annual_recovery = rebalance_df['MonthlyRecovery'].sum() * 12
avg_monthly_per_venue = rebalance_df['MonthlyRecovery'].mean()

print(f"💡 Potential Annual Profit Recovery: ${total_annual_recovery:,.2f} / year across {len(rebalance_df)} venues.")
print(f"💰 Average Monthly Profit Uplift: ${avg_monthly_per_venue:,.2f} / month per venue.")

---
## 9. Conclusion & Key Takeaways

1. **Top-line volume is deceptive:** Expanding on delivery aggregators without healthy unit economics accelerates losses.
2. **Archetype Clarity:** 56.3% of Auckland restaurants (*Optimize*) have the unit economics to justify expansion capital, while 35.0% (*Hold / Stabilize*) need operational restructuring first.
3. **Channel Ownership:** Proprietary self-delivery offers an average **+8.6 percentage point** margin premium over aggregators.
4. **Interactive Application:** The entire engine is deployed and operational at [https://restgrowth.streamlit.app/](https://restgrowth.streamlit.app/).